# County Rainfall and Planting Advisory

**Author:** Leah Atieno J.  
**Cohort:** 1MillionDevs Cohort  

This notebook cleans a hand-typed Machakos rainfall record, combines it with four clean county records into a 5 × 12 NumPy array, and calculates totals, wettest months, dry months and planting windows for an NGO that sends SMS planting advice to farmers.

### Data 

In [1]:
import csv
import numpy as np

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]
months = np.array(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]


## Task 1: Clean the Machakos data
Each text value is converted to an integer inside `try/except ValueError`. `int()` raises `ValueError` for `"N/A"` and `""`, and it safely handles `" 20 "` because it ignores surrounding spaces. Negative rainfall is impossible, so it is also rejected as invalid. Invalid entries are replaced with the rounded average of the valid Machakos values, and the imputed months are printed. A loop is appropriate here because the values are text that must be converted one at a time.

In [2]:
def to_int_or_none(value):
    """Return the value as a non-negative int, or None if it is invalid."""
    try:
        number = int(value)          # raises ValueError for "N/A" and ""
        if number < 0:
            raise ValueError("rainfall cannot be negative")
        return number
    except ValueError:               # specific exception, not a bare except
        return None


def clean_row(raw_values):
    """Convert raw text to ints; impute invalid entries with the rounded mean of valid ones."""
    parsed = [to_int_or_none(v) for v in raw_values]
    valid = [v for v in parsed if v is not None]
    fill_value = round(sum(valid) / len(valid))
    imputed_positions = [i for i, v in enumerate(parsed) if v is None]
    cleaned = [fill_value if v is None else v for v in parsed]
    return cleaned, imputed_positions, fill_value


machakos_clean, imputed_positions, fill_value = clean_row(machakos_raw)

print("Valid values used for the average:", len(machakos_raw) - len(imputed_positions))
print("Imputation value (rounded mean):", fill_value, "mm")
print("Imputed months:", [str(months[i]) for i in imputed_positions])
print("Cleaned Machakos row:", machakos_clean)

Valid values used for the average: 10
Imputation value (rounded mean): 68 mm
Imputed months: ['Apr', 'Sep']
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## Task 2: Combine into a 5 × 12 NumPy array
Rows are counties (in the order of `counties`) and columns are months (January to December).

In [3]:
rain = np.array(clean_rows + [machakos_clean])

print(rain)
print("Shape:", rain.shape)
print("dtype:", rain.dtype)

[[ 64  56  92 219 176  35  18  24  31  61 165 104]
 [ 35  42  70 145 120  78  96 105  72  55  60  38]
 [ 58  90 150 240 180  95  70  85  90  88 130  95]
 [ 30  15  55 150 240 110  90  65  60  90 100  70]
 [ 48  35  88  68 140  20   9  12  68  45 182  96]]
Shape: (5, 12)
dtype: int64


## Task 3: Annual totals and monthly averages
`axis=1` collapses the columns (months), giving one total per county. `axis=0` collapses the rows (counties), giving one average per month.

In [4]:
annual_totals = rain.sum(axis=1)
monthly_avg = rain.mean(axis=0)

for county, total in zip(counties, annual_totals):
    print(f"{county:<9} annual total: {total} mm")

print()
print("Average rainfall per month across all five counties (mm):")
for month, avg in zip(months, monthly_avg):
    print(f"  {month}: {avg:.1f}")

Nairobi   annual total: 1045 mm
Nakuru    annual total: 916 mm
Kisumu    annual total: 1371 mm
Mombasa   annual total: 1075 mm
Machakos  annual total: 811 mm

Average rainfall per month across all five counties (mm):
  Jan: 47.0
  Feb: 47.6
  Mar: 91.0
  Apr: 164.4
  May: 171.2
  Jun: 67.6
  Jul: 56.6
  Aug: 58.2
  Sep: 64.2
  Oct: 67.8
  Nov: 127.4
  Dec: 80.6


## Task 4: Wettest month per county
`np.argmax(axis=1)` returns the column position of the largest value in each row, and indexing the `months` array with those positions converts them to month names. If two months tie, `argmax` returns the first (earliest) one; no ties occur in this data.

In [5]:
wettest_idx = np.argmax(rain, axis=1)
wettest_names = months[wettest_idx]

for county, month, mm in zip(counties, wettest_names, rain.max(axis=1)):
    print(f"{county:<9} wettest month: {month} ({mm} mm)")

Nairobi   wettest month: Apr (219 mm)
Nakuru    wettest month: Apr (145 mm)
Kisumu    wettest month: Apr (240 mm)
Mombasa   wettest month: May (240 mm)
Machakos  wettest month: Nov (182 mm)


## Task 5: Dry months (below 50 mm)
A boolean mask `rain < 50` is `True` where a month is dry. Summing along `axis=1` counts the `True` values in each row.

In [6]:
dry_mask = rain < 50
dry_counts = dry_mask.sum(axis=1)

for county, count, mask_row in zip(counties, dry_counts, dry_mask):
    print(f"{county:<9} dry months: {count}  ->  {', '.join(months[mask_row])}")

Nairobi   dry months: 4  ->  Jun, Jul, Aug, Sep
Nakuru    dry months: 3  ->  Jan, Feb, Dec
Kisumu    dry months: 0  ->  
Mombasa   dry months: 2  ->  Jan, Feb
Machakos  dry months: 6  ->  Jan, Feb, Jun, Jul, Aug, Oct


## Task 6: Planting window (two consecutive months with the highest combined rainfall)
`rain[:, :-1] + rain[:, 1:]` adds each month to the month after it, giving 11 consecutive-pair totals per county. `argmax` finds the best pair, and the window is named by its start and end months. December-to-January is not treated as a pair because the data covers a single calendar year.

In [7]:
pair_totals = rain[:, :-1] + rain[:, 1:]          # shape (5, 11)
start_idx = np.argmax(pair_totals, axis=1)
planting_windows = np.char.add(np.char.add(months[start_idx], "-"), months[start_idx + 1])
window_totals = pair_totals.max(axis=1)

for county, window, total in zip(counties, planting_windows, window_totals):
    print(f"{county:<9} planting window: {window} ({total} mm combined)")

Nairobi   planting window: Apr-May (395 mm combined)
Nakuru    planting window: Apr-May (265 mm combined)
Kisumu    planting window: Apr-May (420 mm combined)
Mombasa   planting window: Apr-May (390 mm combined)
Machakos  planting window: Nov-Dec (278 mm combined)


## Task 7: Write `rainfall_summary.csv`
The file is written with `csv.DictWriter` and then read back to confirm it has five rows and the correct columns.

In [8]:
fieldnames = ["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"]

rows = [
    {
        "county": county,
        "annual_total_mm": int(total),
        "wettest_month": str(wettest),
        "dry_months": int(dry),
        "planting_window": str(window),
    }
    for county, total, wettest, dry, window
    in zip(counties, annual_totals, wettest_names, dry_counts, planting_windows)
]

with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

# Read back to verify
with open("rainfall_summary.csv", newline="") as f:
    check = list(csv.DictReader(f))

print("Columns:", list(check[0].keys()))
print("Data rows:", len(check))
for row in check:
    print(row)

Columns: ['county', 'annual_total_mm', 'wettest_month', 'dry_months', 'planting_window']
Data rows: 5
{'county': 'Nairobi', 'annual_total_mm': '1045', 'wettest_month': 'Apr', 'dry_months': '4', 'planting_window': 'Apr-May'}
{'county': 'Nakuru', 'annual_total_mm': '916', 'wettest_month': 'Apr', 'dry_months': '3', 'planting_window': 'Apr-May'}
{'county': 'Kisumu', 'annual_total_mm': '1371', 'wettest_month': 'Apr', 'dry_months': '0', 'planting_window': 'Apr-May'}
{'county': 'Mombasa', 'annual_total_mm': '1075', 'wettest_month': 'May', 'dry_months': '2', 'planting_window': 'Apr-May'}
{'county': 'Machakos', 'annual_total_mm': '811', 'wettest_month': 'Nov', 'dry_months': '6', 'planting_window': 'Nov-Dec'}


## Planting Advisory
Kisumu is the wettest county (1,371 mm a year) and Machakos is the driest (811 mm), so farmers in Machakos should plan around water shortage more carefully than those elsewhere. For Nairobi, Nakuru, Kisumu and Mombasa the best planting window is April to May, so farmers there should have seed and land ready by the end of March. Machakos farmers should plant in November to December, the county's wettest two months, and should treat the rest of the year as comparatively dry and unreliable. Dry months (under 50 mm) are most common in Machakos (6) and Nairobi (4), so farmers in those counties should use drought-tolerant varieties and mulching or water storage between June and September. **Caveat:** two of the twelve Machakos readings (April and September) were missing and were estimated at 68 mm, so Machakos advice is less certain than for the other counties.

## Data Quality Notes
- **What was wrong:** Two of the twelve Machakos entries were unusable: April was `"N/A"` and September was an empty string `""`. One more entry, June, had extra spaces (`" 20 "`) but was valid after conversion, so it was kept as 20 mm.
- **What was done:** Invalid entries were caught with a specific `except ValueError` and replaced with the rounded mean of the 10 valid Machakos readings (675 / 10 = 67.5, rounded to 68 mm). The imputed months (April and September) are printed in Task 1.
- **Effect on results:** Imputing 68 mm in both months adds 136 mm to the Machakos annual total (811 mm in total), and neither imputed month is counted as dry (below 50 mm). The wettest month (November, 182 mm) and the planting window (November to December) would change only if the real April or September reading was unusually high (above 182 mm for the wettest month, or an April reading above about 138 mm for the window). The annual total and the dry-month count are directly affected by the imputed values, so Machakos figures should be treated as estimates.
- **Not checked:** The four other counties were assumed to be correct. Their readings were not validated against the original station records.
- **Recommendation:** Re-enter the Machakos April and September readings from the station logs, then re-run this notebook.